# MNIST 手写识别优化笔记

In [ ]:
import torch
from torchvision import transforms,datasets

## 1. 问题现象与根本原因

### 现象

用自己手写的数字图片测试，"7" 和 "9" 经常识别错，而 MNIST 测试集上准确率却很高。

### 根本原因：Distribution Shift（分布偏移）

这是机器学习中最常见、也最容易忽视的问题。

**核心原则：**

> 模型只会它见过的东西。训练数据和真实输入的分布越接近，效果越好。

| 对比维度 | MNIST 训练/测试集    | 你自己的手写图片       |
| -------- | -------------------- | ---------------------- |
| 背景     | 纯黑，干净           | 纸张纹理、光影、噪点   |
| 数字风格 | 美式写法，扫描处理过 | 你自己的笔迹，手机拍摄 |
| 图像来源 | 标准化数据集         | 现实环境，质量不稳定   |

MNIST 测试集和训练集来自**同一个分布**，所以测试准确率高。
你的手写图片和训练集来自**不同分布**，模型从没见过这种图，自然会出错。

这两件事可以同时成立，不矛盾。

## 2. 改动一：数据增强

### 改了什么
**改之前：**

In [ ]:
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,),(0.3081,))
])

**改之后：**

In [ ]:
transform=transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0,translate=(0.1,0.1),shear=10),
    #degree旋转角度    translate 平移范围(x,y)   shear错切(一个值对x轴错切，四个值对x,y轴错切）
    transforms.ToTensor(),
    transforms.Normalize((0.1307,),(0.3081,))
])

### 为什么这样改

**数据增强（Data Augmentation）** 的本质是：在训练时人为制造数据的多种变体，让模型提前见过各种形态，从而提升泛化能力。

```
原始图：正放的 "7"
↓ RandomRotation
训练图：倾斜了 -12° 的 "7"、倾斜了 +8° 的 "7"、正放的 "7" ...
```

每个 epoch 每张图都会被随机变换，相当于把 60,000 张图变成了近乎无限多的变体。模型见过各种角度的 "7"，真实照片里稍微斜一点的 "7" 就不会认错了。

### 两个增强操作的含义

**`transforms.RandomRotation(15)`**

随机旋转 ±15°。

- 对抗：你写字时的倾斜角度
- 为什么是 15°：MNIST 数字通常不会旋转超过 20°，太大反而让模型学偏

**`transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=10)`**

仿射变换，包含两个操作：

- `translate=(0.1, 0.1)`：随机平移，幅度最多是图片宽/高的 10%
	- 对抗：你的数字没写在正中央
- `shear=10`：随机错切（让数字"倾斜剪切"）
	- 对抗：你写字时笔画的倾斜方向

### 直觉图示

```
不加增强：模型见过的 "9"      你手写的 "9"
          ○                   ○
          |                    \   ← 稍微斜了，模型蒙了

加增强：  模型见过的 "9"
          ○  ○  ○  ∘
          |   \  /  |          全见过，认出来了
```

---

##

## 3. 改动二：训练集与测试集使用不同 transform

### 改了什么

In [ ]:
# 改之前：训练集和测试集用同一个 transform
train_dataset = datasets.MNIST(..., transform=transform)
test_dataset  = datasets.MNIST(..., transform=transform)

# 改之后：分开
transform      = transforms.Compose([RandomRotation, RandomAffine, ToTensor, Normalize])  # 训练用，带增强
transform_test = transforms.Compose([ToTensor, Normalize])                                  # 测试用，不增强

train_dataset = datasets.MNIST(..., transform=transform)
test_dataset  = datasets.MNIST(..., transform=transform_test)  # 只用 transform_test

### 为什么测试集不能用增强

这是一个**逻辑问题**，不是性能问题。

测试集的目的是：**评估模型在"正常输入"下的真实水平**。

如果测试集也随机旋转，你每次测的都是不同角度的图，相当于每次考试题目都在随机变，测出来的分数没有可比性，不能真实反映模型能力。

更重要的是：你用模型预测真实图片时，图片是固定的，不会被随机旋转。测试集应该尽量模拟"真实使用场景"。

**规则：数据增强只加在训练集。**

## 4. 改动三：Otsu 自适应去噪阈值

### 改了什么

**改之前（固定阈值）：**

```python
arr = np.where(arr > 50, arr, 0.0)   # 凡是亮度低于 50 的像素，全部变成 0（黑）
```

**改之后（Otsu 自适应阈值）：**

```python
thresh = _otsu_threshold(arr)         # 自动计算最佳阈值
arr = np.where(arr > thresh, arr, 0.0)
```

### 为什么固定阈值 50 有问题

去噪的目的是：**把背景噪点压成纯黑，保留笔画。**

问题在于，不同图片的亮度范围完全不同：

```
用马克笔拍照：笔画最亮 ≈ 240，背景噪点 ≈ 30     → 阈值50 刚好，效果好
用铅笔拍照：  笔画最亮 ≈ 120，背景噪点 ≈ 20     → 阈值50 会把大半笔画也抹掉！
光线暗的环境：笔画最亮 ≈ 90，                    → 阈值50 只剩一点残影
```

固定阈值无法适应不同照片，会出现"把 7 的横杠抹掉，导致 7 变成 1"这类问题。

### Otsu 算法原理

Otsu（大津算法）是图像处理中的经典算法，思路是：

> **找一个阈值 t，使得把像素分成"亮组（笔画）"和"暗组（背景）"之后，两组内部的方差之和最小（即组间方差最大）。**

直觉理解：

- 好的阈值 → 笔画像素紧密聚在一起，背景像素紧密聚在一起，两组泾渭分明
- 坏的阈值 → 某一组内部很分散，说明阈值割错了位置

```
像素亮度直方图（示意）：

数量
 │   背景峰                 笔画峰
 │   ████                   ████
 │   ████                   ████
 │   ████    ← Otsu 找到 →  ████
 └────────────────────────────── 亮度
 0   低                     高
```

Otsu 会自动找到两个峰之间的谷底作为阈值，不受整体亮度高低影响。

### 代码逻辑（可以读懂的版本）

```python
def _otsu_threshold(arr):
    # 统计每个亮度值（0-255）出现了多少次
    hist, _ = np.histogram(arr.flatten(), bins=256, range=(0, 256))

    # 穷举所有可能的阈值 t（0~255），找让组间方差最大的那个
    for t in range(256):
        # 把像素分成 ≤t（背景）和 >t（笔画）两组
        # 计算两组的均值差 × 各组权重 = 组间方差
        # 组间方差越大 → 两组分得越开 → 阈值越好
        ...

    return best_thresh   # 返回最佳阈值
```

---

## 5. 一个重要教训：准确率高 ≠ 真实效果好

### 发生了什么

第一轮修改（加数据增强 + 改阈值为 `arr.max() * 0.20`）之后：

- MNIST 测试集准确率从 ~99% 提升到 **99.42%**（更高了）
- 但所有手写图片全部被预测成 **8**（更差了）

这两件事**同时发生**，不矛盾。

### 为什么

**准确率高**：数据增强让模型在 MNIST 分布内泛化更好，测试集（同分布）自然分更高。

**全变成 8**：`arr.max() * 0.20` 这个阈值在照片对比度低时会变得很小（比如最大亮度只有 100，阈值就是 20），导致大量背景噪点没被清除，模型看到的是一堆杂乱的点。

而 **8 是 0-9 中形状最"复杂"的数字**（两个封闭圆），杂乱的噪点图案天然容易和它匹配。

### 核心认知

```
测试集准确率 = 模型在"MNIST分布"上的表现
手写预测效果 = 模型在"你的照片分布"上的表现

这两个是不同的指标，可以同时往相反方向变化。
```

**在实际工程中，评估指标必须和真实使用场景对齐。** 如果你的模型要用于识别手机拍摄的手写数字，就要用手机拍摄的手写数字来测，而不是 MNIST 测试集。

---

## 6. 调试方法：怎么快速定位是预处理还是模型的问题

`predict_image` 函数输出三张图：

```
[原始图片]  →  [模型实际看到的 28×28 图]  →  [概率分布]
```

**第二张图（Model Input）是最重要的调试工具。**

| 第二张图的样子           | 问题所在                     | 怎么修                      |
| ------------------------ | ---------------------------- | --------------------------- |
| 全是杂乱噪点             | 预处理：背景没清干净         | 调整阈值（用 Otsu）         |
| 数字变形/缺笔画          | 预处理：阈值太高把笔画抹掉了 | 降低阈值                    |
| 数字看起来正常，但预测错 | 模型：没见过这种风格         | 加数据增强，重新训练        |
| 数字偏在角落，没居中     | 预处理：裁剪或填充逻辑有问题 | 检查 padding 和 canvas 逻辑 |

**调试原则：先看中间结果，再猜问题在哪。** 不要一上来就怀疑网络参数。

---

## 7. 总结：以后遇到类似问题的自查清单

### 当"训练准确率高，但预测自己的图片效果差"时

```
□ 1. 看 Model Input（第二张图）
      → 图很乱？  → 预处理问题，去噪阈值太低
      → 图缺笔画？→ 预处理问题，阈值太高
      → 图看起来正常？→ 继续往下查

□ 2. 检查 Distribution Shift
      → 你的图片和训练集的风格差距有多大？
      → 是否需要加数据增强？

□ 3. 检查 transform 是否分离
      → 训练集：带增强
      → 测试集 / 推理时：不带增强
```

### 以后想自己改进时的方向

| 方向           | 具体操作                               | 预期效果               |
| -------------- | -------------------------------------- | ---------------------- |
| 增强更强       | `RandomRotation(20)`, `shear=15`       | 对倾斜手写更鲁棒       |
| 增加对比度增强 | `transforms.ColorJitter(contrast=0.5)` | 对光照不均的照片更鲁棒 |
| 更换优化器     | `optim.Adam(lr=0.001)`                 | 通常比 SGD 更容易收敛  |
| 训练更多 epoch | `num_epochs=20`                        | 如果还有欠拟合空间     |
| 收集自己的数据 | 拍几百张自己写的数字，加入训练         | 从根本上缩小分布偏移   |

### 三条普适原则

1. **评估集要和真实场景分布一致**，否则评估数字没意义
2. **先看中间结果再猜原因**，调试从可视化开始
3. **数据增强是缩小训练-真实差距的最简单方法**，但不是万能的；真正的解决办法是让训练数据和真实数据尽量来自同一分布